# EX_08 — Introducción a agentes (ejercicios)

**Notebook de referencia:** `notebook/08_Introduccion_Agentes.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Definir 2 herramientas

Escribe funciones Python puras `get_time_utc()` (puede ser fake) y `hash_text(s: str)` (usa `hashlib.sha256` en hex). Estas serán tus "tools".


In [1]:
import hashlib
from datetime import datetime, timezone

def get_time_utc() -> str:
    """Devuelve el timestamp actual en formato ISO bajo la zona horaria UTC."""
    # Obtenemos el tiempo actual con la timezone UTC integrada de forma precisa
    return datetime.now(timezone.utc).isoformat()

def hash_text(s: str) -> str:
    """Genera el hash SHA-256 de un texto dado y lo devuelve en formato hexadecimal."""
    # Encodificamos el string a bytes para que hashlib pueda procesarlo
    return hashlib.sha256(s.encode('utf-8')).hexdigest()


## Actividad 2 — Cuándo usar tool

Para cada intención del usuario (`"What time is it?"`, `"Digest of hello"`), escribe en comentarios si el LLM debería llamar tool o responder directo.


In [2]:
# TODO: your comments per intent

# Intención 1: "What time is it?"
# Decisión: LLAMAR A TOOL (get_time_utc)
# Razón: El LLM es un modelo estático y no tiene acceso a un reloj interno en tiempo real. 
# Si responde de forma directa, alucinará la hora exacta actual o se disculpará.

# Intención 2: "Digest of hello"
# Decisión: LLAMAR A TOOL (hash_text)
# Razón: Aunque un LLM "sabe" qué es un hash SHA-256, los modelos de lenguaje son pésimos 
# haciendo criptografía y cálculos matemáticos a nivel de bytes directos. 
# Forzarlo a responder directamente provocaría casi con total seguridad un hash erróneo.


## Actividad 3 — Bucles

En español (celda markdown), explica el riesgo de **bucles infinitos** tool→modelo→tool y una mitigación (límite de pasos, detector de repetición).


_Tu explicación:_

El riesgo de bucles infinitos (Tool $\rightarrow$ Modelo $\rightarrow$ Tool):Un bucle infinito en un agente ocurre cuando el LLM entra en un estado de confusión o "miopía" algorítmica. Por ejemplo:El usuario hace una petición.El modelo determina que necesita una tool para resolverla y la ejecuta.Al recibir el resultado de la herramienta, el LLM no sabe cómo interpretarlo o el formato recibido no colma las expectativas de su prompt del sistema.En lugar de finalizar, el modelo vuelve a llamar exactamente a la misma tool (o a otra distinta) repitiendo los mismos parámetros una y otra vez de forma cíclica.Esto genera un gasto masivo de tokens (dinero), latencia y puede llegar a saturar las APIs externas por la velocidad de las peticiones concurrentes.Estrategias de mitigación:Límite de pasos (Max Iterations Limit): Es la defensa más robusta y simple. Consiste en configurar un contador estricto en el bucle del agente (por ejemplo, un máximo de 5 o 10 iteraciones). Si el agente llega al paso límite sin haber devuelto una respuesta final, el entorno de ejecución detiene forzosamente el flujo y devuelve un error controlado o un resumen de lo que pudo procesar.Detector de repetición (Repetitive Action Detection): Consiste en guardar un registro histórico de los pasos previos dados por el agente dentro del mismo turno de conversación. Si el sistema detecta que el modelo está solicitando ejecutar exactamente la misma función con los mismos argumentos de entrada por segunda o tercera vez consecutiva, intercepta la ejecución y penaliza/modifica el prompt dinámicamente para exigirle que tome una ruta de salida o acepte el fracaso.